# Modelos de churn y segmentación de clientes

**Equipo:** Modelos · **Dataset:** `telco_churn_clean.csv`

Este notebook compara dos clasificadores para predecir `Churn` y crea segmentos con K-Means. Se conserva intacto el CSV entregado por Limpieza y EDA. La preparación supervisada vive dentro de un `Pipeline` para que el escalado y la codificación se ajusten solo con los datos de entrenamiento.

**Evaluación:** split estratificado 80/20, semilla fija; mostramos el baseline con umbral 0.50 y ajustamos un umbral para cada modelo solo con validación cruzada del train. Reportamos precision, recall, F1 y matriz de confusión para `Yes`. F1 de la clase `Yes` es el criterio principal porque equilibra precision y recall; precision y recall se reportan para hacer visible el intercambio.

> Las relaciones y probabilidades describen este dataset; no demuestran causas ni garantizan el comportamiento futuro.

## 1. Librerías y rutas

Si falta una librería en el entorno de VS Code, instálala en el entorno virtual del proyecto: `python3 -m pip install pandas numpy matplotlib seaborn scikit-learn jupyter`. En Mac, selecciona ese entorno como kernel del notebook.

In [ ]:
from pathlib import Path
import warnings
import sys
import json
import joblib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, f1_score, precision_score, recall_score, silhouette_score,
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
NOTEBOOK_CWD = Path.cwd().resolve()
REPO_ROOT = next((root for root in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (root / ".git").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("No encontré la raíz del repositorio Git. Abre el notebook desde una copia del proyecto.")

PROJECT_DIR = REPO_ROOT
sys.path.insert(0, str(PROJECT_DIR))
from src.modeling.feature_engineering import build_model_features
DATA_PATH = PROJECT_DIR / "data" / "processed" / "telco_churn_clean.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"No encontré el CSV limpio en {DATA_PATH}. Actualiza la rama con los cambios del equipo de Limpieza y EDA."
    )

OUTPUT_DIR = PROJECT_DIR / "data" / "output"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Datos: {DATA_PATH}")
print(f"Salidas: {OUTPUT_DIR}")


## 2. Cargar y validar datos

`customerID` identifica al cliente y no es una señal predictiva. `Churn` es el objetivo; la convertimos a 1 para `Yes` y 0 para `No`. Las filas duplicadas por identificador deben revisarse antes de entrenar.

In [ ]:
df = pd.read_csv(DATA_PATH)

assert len(df) == 7043, f"Se esperaban 7043 filas; hay {len(df)}"
assert "customerID" in df.columns and "Churn" in df.columns
assert df["customerID"].is_unique, "customerID tiene duplicados"
assert df.isna().sum().sum() == 0, "El CSV contiene nulos"
assert set(df["Churn"].unique()) == {"Yes", "No"}

y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
print(f"Dimensiones: {df.shape}; tasa churn: {y.mean():.1%}")
df["Churn"].value_counts().to_frame("clientes").assign(porcentaje=lambda x: x["clientes"] / len(df))


## 3. Separación train/test y preprocesamiento

El conjunto de prueba se reserva hasta la evaluación. One-hot encoding y escalado se ajustan dentro de cada pipeline con `X_train` solamente. Las categorías nuevas que aparezcan en test se ignoran de forma segura.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

X_ENGINEERED = build_model_features(X)
numeric_features = X_ENGINEERED.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_ENGINEERED.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Proporción churn train/test:", round(y_train.mean(), 3), round(y_test.mean(), 3))


## 4. Entrenar y comparar clasificadores

Comparamos regresión logística como referencia interpretable y Random Forest como modelo no lineal. Ambos usan los mismos datos y preprocesamiento. El peso de clases se deja inicialmente sin ajustar para que las probabilidades y el umbral 0.50 sean una referencia clara; si el recall queda bajo, se puede ajustar el umbral usando solo validación de entrenamiento y volver a reportar el test una sola vez.

In [ ]:
models = {
    "Regresión logística": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, min_samples_leaf=2, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
}

fitted_models = {}
rows = []
for name, estimator in models.items():
    pipeline = Pipeline([
        ("feature_engineering", FunctionTransformer(build_model_features, validate=False)),
        ("preprocess", preprocessor),
        ("model", estimator),
    ])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    fitted_models[name] = pipeline
    rows.append({
        "modelo": name,
        "precision_yes": precision_score(y_test, pred, zero_division=0),
        "recall_yes": recall_score(y_test, pred, zero_division=0),
        "f1_yes": f1_score(y_test, pred, zero_division=0),
    })

metrics = pd.DataFrame(rows).set_index("modelo").sort_values("f1_yes", ascending=False)
metrics.round(3)


In [ ]:
for name, pipeline in fitted_models.items():
    pred = pipeline.predict(X_test)
    print(f"\n{name} — matriz de confusión @ umbral 0.50 (filas=real; columnas=predicción; orden No, Yes)")
    print(confusion_matrix(y_test, pred, labels=[0, 1]))


In [ ]:
# Cada modelo se ajusta con su propio umbral, elegido solo con OOF del train; test queda reservado.
threshold_grid = np.linspace(0.05, 0.80, 151)
threshold_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
tuned_thresholds = {}
tuned_rows = []
tuned_train_rows = []
tuned_test_predictions = {}

for name, pipeline in fitted_models.items():
    train_oof_proba = cross_val_predict(
        pipeline, X_train, y_train, cv=threshold_cv, method="predict_proba", n_jobs=1
    )[:, 1]
    threshold = max(
        threshold_grid,
        key=lambda value: f1_score(y_train, train_oof_proba >= value, zero_division=0),
    )
    tuned_thresholds[name] = float(threshold)
    oof_pred = (train_oof_proba >= threshold).astype(int)
    tuned_train_rows.append({
        "modelo": name, "threshold": threshold,
        "precision_yes": precision_score(y_train, oof_pred, zero_division=0),
        "recall_yes": recall_score(y_train, oof_pred, zero_division=0),
        "f1_yes": f1_score(y_train, oof_pred, zero_division=0),
    })
    test_proba = pipeline.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= threshold).astype(int)
    tuned_test_predictions[name] = test_pred
    tuned_rows.append({
        "modelo": name, "threshold": threshold,
        "precision_yes": precision_score(y_test, test_pred, zero_division=0),
        "recall_yes": recall_score(y_test, test_pred, zero_division=0),
        "f1_yes": f1_score(y_test, test_pred, zero_division=0),
    })

tuned_train_metrics = (
    pd.DataFrame(tuned_train_rows).set_index("modelo").sort_values("f1_yes", ascending=False)
)
tuned_metrics = pd.DataFrame(tuned_rows).set_index("modelo").sort_values("f1_yes", ascending=False)
print("Métricas OOF del train de esta división (para ajustar umbrales):")
display(tuned_train_metrics.round(3))
print("Métricas del test (solo evaluación):")
tuned_metrics.round(3)


In [ ]:
for name, pred in tuned_test_predictions.items():
    print(f"\n{name} — matriz de confusión @ umbral {tuned_thresholds[name]:.2f} (filas=real; columnas=predicción; orden No, Yes)")
    print(confusion_matrix(y_test, pred, labels=[0, 1]))


### Decisión del modelo supervisado

La variable objetivo es `Churn = Yes`. F1 es el criterio principal; cuando dos opciones tienen F1 muy parecido, revisamos recall porque indica qué proporción de los churners reales detecta el modelo.

**Reducción de variables.** El CSV limpio permanece intacto. El modelo resume los seis servicios de internet en dos conteos: `SecuritySupportCount` (seguridad, respaldo, protección de dispositivos y soporte) y `StreamingServicesCount` (TV y películas). El pipeline transforma los 19 campos originales en 15 variables. En la comparación OOF de candidatas, la regresión logística completa obtuvo F1 0.637 y recall 0.740; agrupando los servicios obtuvo F1 0.636 y recall 0.779. La caída de F1 es 0.001 y el recall aumenta 3.9 puntos, por eso adoptamos esta agrupación.

**Elección del clasificador.** Comparamos ambos clasificadores en cinco divisiones estratificadas. En cada división, el umbral se ajustó usando OOF del train. Random Forest obtuvo F1 medio 0.629 (desviación estándar 0.023), recall medio 0.764 (0.019) y precision media 0.535 (0.026). La regresión logística obtuvo F1 medio 0.628 (0.015), recall 0.759 (0.029) y precision 0.538 (0.030). Están casi empatados; siguiendo la regla acordada, seleccionamos Random Forest por su recall medio ligeramente mayor. La diferencia es pequeña frente a la variación entre divisiones y no implica una ventaja fuerte.

El umbral de producción es la mediana de los cinco umbrales OOF de Random Forest: `0.460`. En un holdout final separado (semilla 47), obtuvo precision 0.508, recall 0.749 y F1 0.605, con matriz `[[764, 271], [94, 280]]` (filas reales, columnas predichas; orden `No`, `Yes`). Ese holdout no se usó para elegir clasificador ni umbral.

La selección por promedio en varias divisiones evita decidir por una sola partición. Aun así, las diferencias son pequeñas; la regresión logística sigue siendo una alternativa más simple e interpretable si el equipo prioriza simplicidad frente a la leve ventaja de recall de Random Forest.

## 5. Segmentación: elección de k y algoritmo

Comparamos K-Means con Gaussian Mixture sobre la misma matriz: `tenure` y `MonthlyCharges` estandarizadas, y `Contract` codificado en indicadores. No usamos `Churn` para formar los clusters; se agrega después para describir cada perfil.

In [ ]:
cluster_features = ["tenure", "MonthlyCharges", "Contract"]
cluster_prep = ColumnTransformer([
    ("num", StandardScaler(), ["tenure", "MonthlyCharges"]),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["Contract"]),
], verbose_feature_names_out=False)

# El script compare_clustering.py ajusta ambos algoritmos con k=2...8 y guarda las métricas.
cluster_comparison = pd.read_csv(OUTPUT_DIR / "clustering_model_comparison.csv")
k_results = cluster_comparison[cluster_comparison["algorithm"] == "K-Means"].copy()
gmm_results = cluster_comparison[cluster_comparison["algorithm"] == "Gaussian Mixture"].copy()
cluster_comparison.query("k in [4, 8]").round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_results["k"], k_results["inertia"], marker="o")
axes[0].axvline(4, color="tab:red", linestyle="--", label="k=4 candidato")
axes[0].set(xlabel="Número de clusters (k)", ylabel="Inertia", title="Método del codo — K-Means")
axes[0].set_xticks(k_results["k"])
axes[0].legend()
axes[1].plot(k_results["k"], k_results["silhouette"], marker="o", label="K-Means")
axes[1].plot(gmm_results["k"], gmm_results["silhouette"], marker="o", label="Gaussian Mixture")
axes[1].axvline(4, color="tab:red", linestyle="--", alpha=0.7)
axes[1].set(xlabel="Número de clusters (k)", ylabel="Silhouette (muestra fija de 2,000)", title="Comparación de silueta")
axes[1].set_xticks(k_results["k"])
axes[1].legend()
plt.tight_layout()
plt.show()

El método del codo muestra que la reducción de inertia se hace menos pronunciada después de `k=4`; los clusters posteriores la siguen bajando, como se espera, pero añaden complejidad. En K-Means, `k=4` obtiene silhouette 0.383, Davies–Bouldin 0.967 y Calinski–Harabasz 4,964.6; `k=8` mejora silhouette a 0.431 y Davies–Bouldin a 0.916, pero Calinski–Harabasz baja a 4,808.2 y el cluster menor representa solo 6.9% de los clientes.

Gaussian Mixture también se evaluó con las mismas variables y `k`. Para `k=4` obtiene silhouette 0.325, Davies–Bouldin 1.234 y Calinski–Harabasz 2,267.8, peores que K-Means con cuatro grupos. Como la silueta, Davies–Bouldin, Calinski–Harabasz y los tamaños favorecen K-Means para esta representación, seleccionamos **K-Means con `k=4`**. El modelo sigue pudiendo asignar segmentos a clientes nuevos. Los criterios internos no garantizan valor comercial; revisamos perfiles y churn después de asignar los grupos.

Las métricas de silueta se estimaron con la misma muestra aleatoria de 2,000 clientes para reducir el tiempo de cálculo. `Churn` se cruza solo después del clustering.

In [ ]:
K_SELECTED = 4  # El codo y la accionabilidad favorecen cuatro perfiles
cluster_pipeline = Pipeline([
    ("preprocess", cluster_prep),
    ("model", KMeans(n_clusters=K_SELECTED, random_state=RANDOM_STATE, n_init=20)),
])
# El pipeline recibe las columnas originales; el ColumnTransformer selecciona solo las tres acordadas.
segments = cluster_pipeline.fit_predict(X)

segmented = df[["customerID", "Churn", "tenure", "MonthlyCharges", "Contract", "InternetService", "PaymentMethod"]].copy()
segmented["segment"] = segments
SEGMENT_NAMES = {
    0: "Relación estable",
    1: "Inicio por afianzar",
    2: "Valor consolidado",
    3: "Retención prioritaria",
}
segmented["segment_name"] = segmented["segment"].map(SEGMENT_NAMES)
segmented["churn_flag"] = y.values

segment_profile = segmented.groupby("segment").agg(
    clientes=("customerID", "size"),
    porcentaje=("customerID", lambda s: len(s) / len(segmented)),
    churn_rate=("churn_flag", "mean"),
    tenure_mediana=("tenure", "median"),
    cargo_mensual_mediana=("MonthlyCharges", "median"),
)
segment_profile["churn_rate"] *= 100
segment_profile["porcentaje"] *= 100
segment_profile.round(1)


In [ ]:
for col in ["Contract", "InternetService", "PaymentMethod"]:
    print(f"\nDistribución de {col} dentro de cada segmento (%)")
    display(pd.crosstab(segmented["segment"], segmented[col], normalize="index").mul(100).round(1))


### Nombres y lectura de los segmentos

El notebook asigna etiquetas descriptivas al k seleccionado después de revisar tamaños y perfiles. Los nombres describen los perfiles observados, no causas del churn. Los IDs de K-Means son arbitrarios; revisa la correspondencia de nombres si cambian datos, semilla, k o versión de scikit-learn.

## 6. Exportar resultados para Power BI/API

El umbral se elige con el train mediante validación cruzada y F1; las métricas finales se calculan en el test reservado. El CSV por cliente usa scores out-of-fold de cinco folds, para que cada fila tenga una puntuación de un modelo que no vio esa fila. Incluye el ID técnico del cluster y un nombre descriptivo de la corrida actual. Los nombres de cluster deben revisarse si cambia la semilla, los datos o la versión de scikit-learn.

Antes de ejecutar el notebook, corre `python src/modeling/compare_final_classifiers.py` para actualizar la comparación de cinco divisiones y el holdout final. Para clientes nuevos, API puede enviar los 19 campos originales. El pipeline final calcula internamente los dos conteos de servicios, transforma las 19 entradas en 15 variables y aplica el modelo. El mismo módulo de ingeniería de variables debe estar disponible al cargar el artefacto.

**Contrato de salida:** `telco_churn_model_output.csv` contiene `customerID`, `churn_probability`, `predicted_churn`, `segment` y `segment_name`. `telco_churn_dashboard_with_models.csv` contiene las columnas originales más estos cuatro campos. Los nombres de segmento y sus IDs se listan en `README_modelos.md`.

In [ ]:
repeated_results = pd.read_csv(OUTPUT_DIR / "final_classifier_repeated_holdout.csv")
classifier_summary = repeated_results.groupby("model").agg(
    f1_mean=("f1_yes", "mean"),
    f1_std=("f1_yes", "std"),
    recall_mean=("recall_yes", "mean"),
    recall_std=("recall_yes", "std"),
    precision_mean=("precision_yes", "mean"),
    precision_std=("precision_yes", "std"),
)
max_mean_f1 = classifier_summary["f1_mean"].max()
near_tie_models = classifier_summary[
    classifier_summary["f1_mean"] >= max_mean_f1 - 0.005
]
BEST_MODEL_NAME = near_tie_models.sort_values(
    ["recall_mean", "f1_mean"], ascending=False
).index[0]
SELECTED_THRESHOLD = float(
    repeated_results.loc[
        repeated_results["model"] == BEST_MODEL_NAME, "threshold_oof"
    ].median()
)
selected_model = fitted_models[BEST_MODEL_NAME]
print(f"Modelo elegido en validación repetida: {BEST_MODEL_NAME}")
print(f"Umbral mediano OOF: {SELECTED_THRESHOLD:.3f}")
display(classifier_summary.round(3))

# Probabilidades OOF de todos los clientes: cada fila se predice sin entrenar en ella.
final_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_probabilities = cross_val_predict(
    selected_model, X, y, cv=final_cv, method="predict_proba", n_jobs=1
)[:, 1]
oof_predictions = (oof_probabilities >= SELECTED_THRESHOLD).astype(int)

model_output = pd.DataFrame({
    "customerID": df["customerID"],
    "churn_probability": oof_probabilities,
    "predicted_churn": oof_predictions,
    "segment": segments,
    "segment_name": segmented["segment_name"],
})
model_output.to_csv(OUTPUT_DIR / "telco_churn_model_output.csv", index=False)
pbi_output = df.merge(model_output, on="customerID", validate="one_to_one")
pbi_output.to_csv(OUTPUT_DIR / "telco_churn_dashboard_with_models.csv", index=False)
print("Guardado:", OUTPUT_DIR / "telco_churn_model_output.csv")
print("Guardado:", OUTPUT_DIR / "telco_churn_dashboard_with_models.csv")
print("Los artefactos de modelos se guardan después del cruce y del resumen de segmentos.")

## 7. Cruce de supervisado, K-Means y churn observado

Este análisis cruza por `customerID` el segmento, la probabilidad OOF y el churn real. Es una descripción posterior para evaluar cómo se concentra el riesgo; `Churn` no alimenta K-Means ni las predicciones. La tasa real del segmento y la probabilidad promedio ayudan a comparar el grupo observado con el score del clasificador.

La columna `parte_de_todos_los_churners` permite ver qué fracción de los clientes que abandonaron pertenecía a cada segmento.

In [ ]:
# Cruce post-hoc: K-Means + riesgo OOF + churn observado. Churn no intervino en K-Means ni en las predicciones OOF.
integrated = df[["customerID", "Churn"]].merge(
    model_output, on="customerID", validate="one_to_one"
)
integrated["churn_flag"] = (integrated["Churn"] == "Yes").astype(int)

segment_risk = integrated.groupby(["segment", "segment_name"]).agg(
    clientes=("customerID", "size"),
    tasa_del_total=("customerID", lambda s: len(s) / len(integrated) * 100),
    churn_real=("churn_flag", "mean"),
    probabilidad_media=("churn_probability", "mean"),
    churners=("churn_flag", "sum"),
).reset_index()
segment_risk["churn_real"] *= 100
segment_risk["probabilidad_media"] *= 100
segment_risk["parte_de_todos_los_churners"] = (
    segment_risk["churners"] / integrated["churn_flag"].sum() * 100
)
segment_risk.to_csv(OUTPUT_DIR / "telco_churn_segment_risk_summary.csv", index=False)
print("Guardado: telco_churn_segment_risk_summary.csv")
segment_risk.round(1)


### Riesgo por segmento y acciones posibles

Estas tasas usan churn observado y probabilidades OOF: cada cliente histórico fue puntuado por un modelo que no entrenó con su fila. `predicted_yes` usa el umbral seleccionado de 0.460. El CSV `telco_churn_segment_risk_summary.csv` deja el resumen disponible para análisis y dashboard.

| Segmento | Churn observado | Riesgo OOF medio | Marcados por el modelo | Parte de todos los churners |
|---|---:|---:|---:|---:|
| Relación estable | 3.7% | 7.7% | 1.7% | 2.3% |
| Inicio por afianzar | 24.8% | 36.6% | 36.9% | 23.1% |
| Valor consolidado | 14.2% | 23.5% | 14.8% | 14.4% |
| Retención prioritaria | 49.9% | 63.5% | 76.4% | 60.2% |

Sugerencias para validar con el equipo comercial:

- **Retención prioritaria:** priorizar contacto temprano para revisar instalación, calidad del servicio y si el plan contratado encaja. Probar opciones de permanencia solo si el cliente valora el beneficio.
- **Inicio por afianzar:** reforzar onboarding y ayudar a resolver dudas de factura, pago y uso del servicio; evaluar incentivos de bajo costo.
- **Valor consolidado:** hacer una revisión proactiva de calidad/valor y ofrecer beneficios solo a clientes con score alto.
- **Relación estable:** mantener atención regular y acciones de fidelidad de bajo costo; evitar descuentos generales.

El segmento no debe ser la única regla de contacto: priorizar también por probabilidad individual. Estas propuestas son hipótesis; el dataset no incluye campañas, costos ni resultados de retención para demostrar que funcionen.

### Limitaciones del análisis

- El dataset es una fotografía histórica; no tiene fechas de eventos, historial de contactos ni resultados de campañas. No permite atribuir causalidad ni calcular retorno de inversión de una acción.
- Random Forest fue apenas superior en cinco divisiones repetidas (F1 medio 0.629 frente a 0.628 para logística); las diferencias están dentro de la variación observada. El holdout final obtuvo F1 0.605, así que el modelo deja errores y necesita monitoreo con datos nuevos.
- El umbral 0.460 se ajustó para F1/recall, no con costos reales de falso positivo, falso negativo o capacidad del centro de contacto. En las salidas históricas se marcaron 2,663 de 7,043 clientes como churn probable.
- `gender` y `SeniorCitizen` siguen entre las variables supervisadas; no se ha evaluado si las tasas de error difieren entre grupos. El score no debe ser la única base para negar ofertas o servicios.
- Los clusters usan solo antigüedad, cargo mensual y contrato. K-Means depende de la escala, el número de grupos y la semilla; las métricas internas y los nombres describen esta corrida, no garantizan segmentos estables ni una campaña eficaz.
- Agrupar los seis servicios reduce la dimensionalidad, pero elimina qué complemento exacto tiene cada cliente. `MonthlyCharges` es un cargo, no una medida de consumo.


In [ ]:
# Artefactos para API: ambos pipelines aceptan el DataFrame con campos originales.
segment_descriptions = {
    0: "Antigüedad mediana 54 meses; 62% contrato de dos años; cargo mensual mediano 24.95; churn 4%.",
    1: "Antigüedad mediana 8 meses; 75% mes a mes; cargo mensual mediano 25.55; churn 25%.",
    2: "Antigüedad mediana 61 meses; cargo mensual mediano 94.25; 67% fibra; churn 14%.",
    3: "Antigüedad mediana 13 meses; 93% mes a mes; 81% fibra; churn 50%.",
}
segments_info = {
    "algorithm": "KMeans",
    "k": K_SELECTED,
    "columns": cluster_features,
    "random_state": RANDOM_STATE,
    "segments": [
        {
            "id": int(row.segment),
            "name": str(row.segment_name),
            "description": segment_descriptions[int(row.segment)],
            "customers": int(row.clientes),
            "churn_rate": round(float(row.churn_real) / 100, 4),
        }
        for row in segment_risk.itertuples(index=False)
    ],
}
with (MODEL_DIR / "segments_info.json").open("w", encoding="utf-8") as f:
    json.dump(segments_info, f, ensure_ascii=False, indent=2)
joblib.dump(cluster_pipeline, MODEL_DIR / "telco_customer_segments.joblib")

# Reentrenar el modelo final con todos los registros para predecir clientes nuevos.
# Las métricas de test anteriores no cambian; este artefacto es para despliegue.
deployment_model = selected_model.fit(X, y)
MODEL_FILENAME = "telco_churn_random_forest.joblib"
joblib.dump(deployment_model, MODEL_DIR / MODEL_FILENAME)
final_holdout_report = json.loads((OUTPUT_DIR / "final_classifier_holdout.json").read_text())
churn_model_info = {
    "model": BEST_MODEL_NAME,
    "estimator_class": type(models[BEST_MODEL_NAME]).__name__,
    "variant": "Servicios agrupados en dos conteos",
    "feature_engineering_module": "src.modeling.feature_engineering.build_model_features",
    "engineered_features": list(X_ENGINEERED.columns),
    "selection_reason": "F1 medio casi empatado en cinco divisiones; Random Forest tuvo recall medio ligeramente mayor.",
    "repeated_holdout_mean": {
        "f1_yes": float(classifier_summary.loc[BEST_MODEL_NAME, "f1_mean"]),
        "recall_yes": float(classifier_summary.loc[BEST_MODEL_NAME, "recall_mean"]),
    },
    "test_metrics": {
        metric: final_holdout_report[metric]
        for metric in ["precision_yes", "recall_yes", "f1_yes"]
    },
    "test_confusion_matrix_labels_0_1": final_holdout_report["confusion_matrix_labels_0_1"],
    "artifact": f"models/{MODEL_FILENAME}",
    "threshold": SELECTED_THRESHOLD,
    "positive_class": "Yes",
    "input_columns": list(X.columns),
}
with (MODEL_DIR / "churn_model_info.json").open("w", encoding="utf-8") as f:
    json.dump(churn_model_info, f, ensure_ascii=False, indent=2)
print("Artefactos de API guardados en:", MODEL_DIR)


## 8. Resumen y limitaciones

Antes de compartir resultados, completa estas conclusiones con las métricas observadas y la decisión del equipo:

- Modelo supervisado elegido y justificación frente a las métricas de la clase `Yes`.
- Tamaño, perfil y tasa de churn de los ocho segmentos; usar las etiquetas definidas en `README_modelos.md`.
- Cruce entre segmento y probabilidad/ predicción de churn para identificar dónde concentrar revisión.
- Limitaciones: un solo dataset histórico, categorías y patrones específicos de esta empresa, costos de falsos positivos/falsos negativos no suministrados y asociación que no implica causalidad.
- Usar los nombres de archivo, columnas y segmentos definidos en `README_modelos.md` en la API y Power BI.

**Nota:** `predicted_churn` usa el umbral elegido en el train por validación cruzada y no debe considerarse recomendación definitiva de contacto hasta acordar capacidad y costos de retención. Las probabilidades del CSV de salida son out-of-fold; para servir predicciones nuevas, API debe entrenar y guardar un pipeline final con todo el conjunto de entrenamiento y respetar este mismo preprocesamiento.